# Moteur de recherche visuelle

Le notebook explique pas à pas le fonctionnement du projet :

- chargement d'une image,
- prétraitement pour ResNet50,
- extraction d'un embedding visuel,
- chargement de la base d'embeddings,
- recherche des images les plus similaires,
- visualisation des résultats.

L'application Flask dans `app.py` est la version interactive du projet. Ce notebook sert plutôt de support d'explication, de test et de présentation.

## Lien avec la consigne du projet

La consigne demande de construire un prototype de **requête d'image** permettant de retrouver des articles visuellement similaires, par exemple un T-shirt aperçu dans la rue puis recherché sur un site de shopping.

Le projet est organisé autour de deux parties obligatoires :

1. **Système d'encodage d'image** : transformer une image en représentation vectorielle.
2. **Système de requête d'image** : comparer une image requête avec une base d'images indexées pour retrouver les plus similaires.

Dans ce notebook, on démontre ces deux parties avec le dataset de vêtements CODAIT.

## 1. Préparer l'environnement

On commence par retrouver automatiquement le dossier du projet, puis on l'ajoute au `PYTHONPATH` pour pouvoir importer les modules locaux : `encoder.py`, `preprocessing.py` et `search_engine.py`.

In [ ]:
from pathlib import Path
import sys

current = Path.cwd().resolve()
if (current / "app.py").exists():
    PROJECT_DIR = current
elif (current.parent / "app.py").exists():
    PROJECT_DIR = current.parent
else:
    raise RuntimeError("Impossible de trouver le dossier du projet. Lancez le notebook depuis le projet ou le dossier notebooks.")

sys.path.insert(0, str(PROJECT_DIR))
PROJECT_DIR

## 2. Importer les bibliothèques

Le projet utilise principalement :

- `Pillow` pour lire les images,
- `torch` et `torchvision` pour ResNet50,
- `numpy` pour manipuler les embeddings,
- `matplotlib` pour afficher les résultats.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from preprocessing import load_image, get_all_image_paths
from encoder import ImageEncoder

DATASET_DIR = PROJECT_DIR / "clothing-dataset" / "images"
EMBEDDINGS_PATH = PROJECT_DIR / "embeddings" / "embeddings.npy"
INDEX_PATH = PROJECT_DIR / "embeddings" / "index.json"

print("Dataset :", DATASET_DIR)
print("Embeddings :", EMBEDDINGS_PATH)
print("Index :", INDEX_PATH)

## Vérification des fichiers attendus

Avant de lancer les calculs, on vérifie que les éléments essentiels du projet sont présents : dataset, embeddings sauvegardés et index des chemins d'images.

In [ ]:
required_paths = {
    "Dataset CODAIT": DATASET_DIR,
    "Embeddings sauvegardés": EMBEDDINGS_PATH,
    "Index des images": INDEX_PATH,
    "Application Flask": PROJECT_DIR / "app.py",
    "Encodeur d'images": PROJECT_DIR / "encoder.py",
    "Script de génération": PROJECT_DIR / "generate_embeddings.py",
}

for label, path in required_paths.items():
    status = "OK" if path.exists() else "MANQUANT"
    print(f"{label:25s} : {status} - {path}")

## 3. Explorer rapidement le dataset

Le dataset contient des images de vêtements. Chaque image peut être transformée en vecteur numérique pour être comparée aux autres.

In [ ]:
image_paths = get_all_image_paths(DATASET_DIR)
print(f"Nombre d'images trouvées : {len(image_paths)}")

image_paths[:5]

Lors de la recherche visuelle de vêtements dans une application de shopping, chaque image de catalogue est indexée de cette manière pour permettre une recherche par image.

Affichons quelques images du dataset pour vérifier visuellement les données.

In [ ]:
sample_paths = image_paths[:8]

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for ax, path in zip(axes.ravel(), sample_paths):
    ax.imshow(Image.open(path).convert("RGB"))
    ax.set_title(Path(path).name[:18], fontsize=8)
    ax.axis("off")

plt.tight_layout()

## 4. Prétraiter une image

Avant de passer dans ResNet50, l'image est :

- convertie en RGB,
- redimensionnée en `224 x 224`,
- convertie en tenseur PyTorch,
- normalisée avec les moyennes et écarts-types ImageNet.

C'est exactement ce que fait `load_image()` dans `preprocessing.py`.

In [ ]:
query_path = Path(image_paths[0])
tensor = load_image(query_path)

print("Image choisie :", query_path)
print("Forme du tenseur :", tensor.shape)

La forme `[1, 3, 224, 224]` signifie :

- `1` image dans le batch,
- `3` canaux couleur RGB,
- hauteur `224`,
- largeur `224`.

## 5. Encoder l'image avec ResNet50

Le modèle ResNet50 pré-entraîné sur ImageNet est utilisé sans sa dernière couche de classification. Au lieu de prédire une classe, il produit un vecteur de caractéristiques de dimension `2048`.

Ce vecteur est l'**embedding visuel** de l'image.

In [ ]:
encoder = ImageEncoder()
embedding = encoder(tensor).numpy()[0]

print("Dimension de l'embedding :", embedding.shape)
print("Extrait du vecteur :", embedding[:10])

À ce stade, on a réalisé la première partie obligatoire : **le système d'encodage d'image**. Une image d'entrée est transformée en vecteur numérique exploitable par un moteur de recherche.

## 6. Charger les embeddings pré-calculés

Le fichier `embeddings.npy` contient tous les vecteurs d'images déjà calculés. Le fichier `index.json` contient les chemins des images dans le même ordre.

Donc :

- ligne `0` de `embeddings.npy` correspond à l'image `0` dans `index.json`,
- ligne `1` correspond à l'image `1`,
- etc.

In [ ]:
embeddings = np.load(EMBEDDINGS_PATH).astype("float32")
with open(INDEX_PATH, "r", encoding="utf-8") as f:
    indexed_paths = json.load(f)

print("Forme de la matrice d'embeddings :", embeddings.shape)
print("Nombre de chemins indexés :", len(indexed_paths))
print("Premier chemin :", indexed_paths[0])

Cette étape répond à l'objectif : **générer les représentations vectorielles pour l'ensemble de données et les enregistrer sur disque**. Le fichier `.npy` contient les vecteurs, tandis que le fichier `.json` permet de retrouver l'image associée à chaque vecteur.

## 7. Rechercher par similarité cosinus

Pour comparer deux images, on compare leurs embeddings. Ici, on utilise la similarité cosinus :

- score proche de `1` : images très similaires,
- score proche de `0` : images peu similaires,
- score négatif : directions opposées dans l'espace vectoriel.

Le principe est le même que dans la route `/search` de `app.py`.

## Similarité cosinus

La similarité cosinus mesure la proximité entre deux vecteurs d'embedding en comparant leur direction, indépendamment de leur norme.

- Elle est particulièrement adaptée aux embeddings d'images, car elle capture la ressemblance sémantique plutôt que l'intensité absolue des valeurs.
- La formule est :

  $$\text{similarité cosinus}(A, B) = \frac{A \cdot B}{\|A\|\,\|B\|}$$

- Dans ce projet, les embeddings sont normalisés, ce qui permet de simplifier le calcul : le produit scalaire normalisé devient directement un score de similarité.
- Un score proche de 1 indique des images très similaires, tandis qu'un score proche de 0 indique des images peu similaires.

In [ ]:
def cosine_search(query_image_path, top_k=6):
    query_tensor = load_image(query_image_path)
    query_vector = encoder(query_tensor).numpy()[0].astype("float32")

    db_norm = embeddings / (np.linalg.norm(embeddings, axis=1, keepdims=True) + 1e-9)
    q_norm = query_vector / (np.linalg.norm(query_vector) + 1e-9)

    scores = db_norm @ q_norm
    top_indices = np.argsort(scores)[::-1][:top_k]

    return [
        {
            "path": indexed_paths[i],
            "score": float(scores[i])
        }
        for i in top_indices
    ]

results = cosine_search(query_path, top_k=6)
results

À ce stade, on a réalisé la deuxième partie obligatoire : **le système de requête d'image**. L'utilisateur fournit une image, le système l'encode, puis il cherche les vecteurs les plus proches dans la base.

## 8. Visualiser les résultats

On affiche l'image requête à gauche, puis les images les plus similaires à droite avec leur score.

In [ ]:
def resolve_project_path(path):
    path = Path(path)
    if path.is_absolute():
        return path
    return PROJECT_DIR / path

def show_search_results(query_image_path, results):
    total = len(results) + 1
    fig, axes = plt.subplots(1, total, figsize=(3 * total, 4))

    axes[0].imshow(Image.open(query_image_path).convert("RGB"))
    axes[0].set_title("Requête", fontsize=10)
    axes[0].axis("off")

    for ax, result in zip(axes[1:], results):
        result_path = resolve_project_path(result["path"])
        ax.imshow(Image.open(result_path).convert("RGB"))
        ax.set_title(f"Score: {result['score']:.3f}", fontsize=9)
        ax.axis("off")

    plt.tight_layout()

show_search_results(query_path, results)

## 10. Variante utilisée dans l'application web

Dans l'application Flask, le même principe est exposé sous forme d'API :

- `POST /upload` : ajoute une image à l'index,
- `POST /search` : reçoit une image requête et retourne les images similaires,
- `GET /stats` : affiche le nombre d'images indexées,
- `GET /image` : sert les images retournées par la recherche.

Le fichier `search_engine.py` contient aussi une version basée sur FAISS. FAISS est utile quand la base devient très grande, car il permet d'accélérer la recherche de voisins similaires. Dans ce prototype, la recherche NumPy par similarité cosinus est suffisante pour démontrer le fonctionnement.

## 11. Cas d'utilisation commerciaux

Ce type de moteur peut être utilisé pour :

- rechercher des vêtements similaires sur un site de shopping,
- retrouver des meubles, accessoires ou pièces visuellement proches,
- détecter des images quasi dupliquées dans un catalogue,
- utiliser les embeddings d'image comme variables dans un modèle de recommandation,
- proposer des produits similaires à partir d'une photo utilisateur.

Le prototype réalisé ici se concentre sur les vêtements, mais la même architecture peut être adaptée à d'autres types d'objets.

## 9. Tester avec une autre image

Vous pouvez changer l'indice ci-dessous pour tester une autre image du dataset. Par exemple : `10`, `100`, `500`, etc.

In [ ]:
test_index = 100
query_path = Path(image_paths[test_index])

results = cosine_search(query_path, top_k=6)
show_search_results(query_path, results)

## Objectifs attendus et réponse du projet

| Objectif demandé | Réponse dans ce projet |
|---|---|
| Configurer un service d'encodage d'images | `encoder.py` utilise ResNet50 pour produire un vecteur de 2048 dimensions. |
| Explorer les techniques de représentations vectorielles | Le notebook montre le prétraitement ImageNet et l'extraction d'embeddings. |
| Générer les représentations du dataset | `generate_embeddings.py` parcourt le dataset et encode chaque image. |
| Enregistrer les vecteurs sur disque | Les fichiers `embeddings/embeddings.npy` et `embeddings/index.json` stockent la base vectorielle. |
| Construire un système de requête d'image | `app.py` et ce notebook comparent l'image requête à la base avec la similarité cosinus. |



## Conclusion

Le moteur de recherche visuelle fonctionne en transformant chaque image en vecteur numérique. La recherche ne se fait donc pas sur le nom du fichier ou sur du texte, mais sur la proximité entre vecteurs.

Résumé du pipeline :

1. Image d'entrée.
2. Prétraitement ImageNet.
3. Extraction d'un embedding avec ResNet50.
4. Comparaison avec les embeddings déjà stockés.
5. Retour des images les plus similaires.

Ce notebook est utile pour expliquer et démontrer le projet. L'application Flask reste l'interface utilisateur principale.

Par rapport à la consigne, le projet respecte donc les points obligatoires :

- construction d'un encodeur d'images,
- génération d'embeddings pour le dataset,
- sauvegarde des embeddings sur disque,
- construction d'un moteur de requête par image,
- visualisation des images les plus similaires.